# 장르 결측 보완 — IMDb + JustWatch(한국 공식 제목 DB)

**용도** — IMDb 장르 미확보 853편(6.8%)을 JustWatch로 보완해 커버 93.2%→99.7%로 올리고, 오매칭을 연도 교차검증으로 걸러내며, 결측이 저인지도 작품 신호임을 검증한다.

**보고서 대응** — 보고서 01장 '장르 결측 처리' 박스 · 사용한 데이터 표의 장르 행(99.7% 확보)

**환경** — Python 3.15 / pandas 3.1.1 / scikit-learn 1.9.1 / LightGBM 4.7.0 / SHAP 0.54.1 (requirements.txt)

**실행 순서** — 00(개요) → 01(수집·결합) → 02(EDA) → 03(장르 보완) → 04(언어장벽) → 05(시장파워·미러) → 06(LightGBM) → 07(보정·SHAP) → 08(데뷔)

In [1]:
# 공통 설정 — 데이터는 원본 저장소에서 읽고, 출력물은 이 폴더에만 기록(원본 보호)
from pathlib import Path
import sys
ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
NBOUT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물")
NBOUT_F = NBOUT / "figures"
for _d in [NBOUT, NBOUT_F, NBOUT / "models", NBOUT / "data" / "processed", NBOUT / "data" / "interim"]:
    _d.mkdir(parents=True, exist_ok=True)
print("데이터 루트 =", ROOT)
print("출력 폴더  =", NBOUT)

데이터 루트 = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset
출력 폴더  = C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\넷플릭스_분석_노트북\출력물


**원본 코드** — `scripts/01_justwatch_genre_fill.py`

In [2]:
# ⚠️ 실행 생략 — 네트워크 수집 단계(요청 간격 0.8초 × 853편, 약 11분)이며
# 결과 파일(data/processed/genre_missing_justwatch.csv)이 이미 존재합니다.
# 코드 확인용으로 아래에 남깁니다. 실행하려면 if False를 True로 바꾸세요.
if False:
    # -*- coding: utf-8 -*-
    # ─────────────────────────────────────────────────────────────────────────────
    # [용도] 01. JustWatch GraphQL로 IMDb 장르 미확보 853편의 장르 수집
    #        → data/processed/genre_missing_justwatch.csv 저장 (요청 간격 0.8초, 재개 지원)
    # [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
    #        외부 API: apis.justwatch.com/graphql (API 키 불필요, 네트워크 연결 필요)
    #        실행: python scripts/01_justwatch_genre_fill.py
    # [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
    # ─────────────────────────────────────────────────────────────────────────────
    import json
    import os
    import ssl
    import time
    import urllib.request

    import pandas as pd
    from pathlib import Path

    ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
    OUT = NBOUT / "data/processed/genre_missing_justwatch.csv"

    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    QUERY = """query {
      searchTitles(source: "/kr", country: "KR", filter: {searchQuery: "%s"}, first: 5) {
        edges { node {
          id objectId objectType
          content(country: "KR", language: "ko") {
            title fullPath originalReleaseYear
            genres { shortName translation(language: "ko") }
          }
        } }
      }
    }"""

    def gql(title):
        esc = title.replace("\\", "\\\\").replace('"', '\\"')
        payload = {"query": QUERY % esc}
        req = urllib.request.Request(
            "https://apis.justwatch.com/graphql",
            data=json.dumps(payload).encode(),
            headers={
                "Content-Type": "application/json",
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/126.0",
                "Origin": "https://www.justwatch.com",
                "Referer": "https://www.justwatch.com/kr",
            })
        with urllib.request.urlopen(req, timeout=30, context=ctx) as r:
            return json.loads(r.read().decode())

    def search_genre(title, netflix_cat):
        d = gql(title)
        edges = d.get("data", {}).get("searchTitles", {}).get("edges", []) or []
        want = "MOVIE" if netflix_cat == "Films" else "SHOW"
        pool = [e["node"] for e in edges if e.get("node")]
        cands = [n for n in pool if n.get("objectType") == want] or pool
        if not cands:
            return "", "", "", ""
        n = cands[0]
        c = n.get("content") or {}
        short = "|".join(g.get("shortName", "") for g in (c.get("genres") or []))
        ko = "|".join((g.get("translation") or "") for g in (c.get("genres") or []))
        return c.get("title", ""), c.get("originalReleaseYear", ""), short, ko

    # ---- 데이터 준비 ----
    c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
    c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
    c_all = pd.concat([c_old, c_new]).drop_duplicates()
    cat = pd.read_csv(ROOT / "data/interim/netflix_titles_catalog.csv")
    catmap = cat.set_index("show_title").netflix_category.to_dict()
    cat_gen = cat.set_index("show_title").imdb_genres.fillna("").astype(str).str.strip()

    titles = sorted(c_all.show_title.unique())
    missing_all = [t for t in titles if cat_gen.get(t, "") == ""]
    print(f"장르 결측: {len(missing_all)}편 / 전체 {len(titles)}편", flush=True)

    # 재개: 성공한 것만 스킵
    done = set()
    if os.path.exists(OUT):
        prev = pd.read_csv(OUT)
        done = set(prev[prev.has_result].show_title)
        print(f"기존 진행분(성공): {len(done)}편", flush=True)

    todo = [t for t in missing_all if t not in done]
    print(f"남은 작업: {len(todo)}편", flush=True)

    rows = []
    t0 = time.time()
    for i, t in enumerate(todo):
        ncat = catmap.get(t, "")
        matched, year, short, ko = "", "", "", ""
        ok = False
        for attempt in range(3):
            try:
                matched, year, short, ko = search_genre(t, ncat)
                ok = bool(short)
                break
            except Exception as e:
                if attempt == 2:
                    print(f"  최종실패 {t}: {type(e).__name__}", flush=True)
                time.sleep(3 + attempt * 4)
        rows.append({"show_title": t, "netflix_category": ncat, "has_result": ok,
                     "matched_title": matched, "matched_year": year,
                     "genres_short": short, "genres_ko": ko})
        if (i + 1) % 50 == 0 or i == len(todo) - 1:
            df = pd.DataFrame(rows)
            if os.path.exists(OUT):
                df = pd.concat([pd.read_csv(OUT), df])
            df.to_csv(OUT, index=False, encoding="utf-8-sig")
            rate = (df.has_result.sum() / len(df) * 100) if len(df) else 0
            print(f"  {i+1}/{len(todo)} | 누적 저장 {len(df)}편 | 매칭률 {rate:.0f}% | 경과 {int(time.time()-t0)}초", flush=True)
        time.sleep(0.8)

    df = pd.read_csv(OUT)
    print(f"\n최종: {len(df)}편 처리, 매칭 성공 {int(df.has_result.sum())}편 ({df.has_result.mean()*100:.0f}%)", flush=True)


**원본 코드** — `scripts/02_justwatch_match_qa.py`

In [3]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 02. JustWatch 수집 결과 품질 검증 — 연도 교차검증 + 제목 유사도로
#        오매칭 의심 건 식별 → genre_missing_justwatch.csv 갱신
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1 — requirements.txt 참고
#        실행: python scripts/02_justwatch_match_qa.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import re

import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
df = pd.read_csv(ROOT / "data/processed/genre_missing_justwatch.csv")
df = df[df.has_result].copy()

# 연도 추출: 원제목에 괄호 연도가 있으면 (예: "A Different World (1987)")
def orig_year(t):
    m = re.search(r"\((\d{4})\)", t)
    return int(m.group(1)) if m else None

df["orig_year"] = df.show_title.map(orig_year)
df["matched_year"] = pd.to_numeric(df.matched_year, errors="coerce")

# 1) 연도 교차검증: 원제 연도와 매칭 연도가 2년 이상 차이나면 오매칭 의심
mask_year = df.orig_year.notna() & df.matched_year.notna()
year_ok = (df.loc[mask_year, "orig_year"] - df.loc[mask_year, "matched_year"]).abs() <= 2
print(f"연도 교차검증 가능: {mask_year.sum()}건 | 연도 불일치(의심): {(~year_ok).sum()}건")

suspicious = df[mask_year & ~year_ok]
print("\n== 연도 불일치 샘플 10 ==")
for _, r in suspicious.head(10).iterrows():
    print(f"  {r.show_title} (원연도 {int(r.orig_year)}) → {r.matched_title} ({int(r.matched_year)}) | {r.genres_ko}")

# 2) 제목 유사도: 정규화한 원제와 매칭제목이 완전히 다르면 한국어화 매칭
def norm(s):
    s = re.sub(r"[\(\[\{].*?[\)\]\}]", "", str(s))
    s = re.sub(r"[^0-9a-zA-Z가-힣]", "", s).lower()
    return s

df["orig_norm"] = df.show_title.map(norm)
df["match_norm"] = df.matched_title.map(norm)
df["similar"] = df.apply(lambda r: (r.orig_norm in r.match_norm) or (r.match_norm in r.orig_norm) or (r.orig_norm == r.match_norm), axis=1)
n_similar = df.similar.sum()
print(f"\n제목 유사(원제 포함): {n_similar}건 | 한국어화 등 유사도 낮음: {len(df)-n_similar}건")
diff = df[~df.similar]
print("== 유사도 낮은 샘플 10 ==")
for _, r in diff.head(10).iterrows():
    print(f"  {r.show_title} → {r.matched_title} | {r.genres_ko}")

# 유사도 낮은 그룹의 연도 검증
d2 = diff[diff.orig_year.notna() & diff.matched_year.notna()]
ok2 = ((d2.orig_year - d2.matched_year).abs() <= 2).mean()
print(f"\n유사도 낮음 중 연도 검증 가능 {len(d2)}건 중 일치 {ok2*100:.0f}%")

# 종합 판단: 연도 불일치(검증 가능한 한) = 오매칭으로 탈락
sus_idx = set(suspicious.index)
final = df[~df.index.isin(sus_idx)]
print(f"\n최종 채택: {len(final)}편 (오매칭 의심 {len(sus_idx)}편 제외)")
final.to_csv(NBOUT / "data/processed/genre_missing_justwatch.csv", index=False, encoding="utf-8-sig")
print("CSV 갱신 완료 (오매칭 의심 제거)")


연도 교차검증 가능: 7건 | 연도 불일치(의심): 0건

== 연도 불일치 샘플 10 ==

제목 유사(원제 포함): 232건 | 한국어화 등 유사도 낮음: 582건
== 유사도 낮은 샘플 10 ==
  #AtFirstSight → 사랑이 머무는 풍경 | 드라마|로맨스
  #Mom, I’m coming home → 커밍 홈 인 더 다크 | 범죄|공포|스릴러
  30 Days Left → 미션 파서블 인 30 데이즈 | 액션|코미디|범죄|유럽에서 제작
  30 for 30: O.J.: Made in America → O.J : 메이드 인 아메리카 | 다큐멘터리
  5 Ways To Socially Exclude My Husband → 남편을 사회적으로 말살하는 다섯 가지 방법 | 드라마|스릴러
  A Fist Full of Lead → 골드체이서: 최후의 추격자 | 드라마|서부
  A Life for the Self-Taught → A Self-Educated Life | 코미디
  A Lot Like Paris → 우리, 사랑일까요? | 코미디|드라마|로맨스|유럽에서 제작
  A Nation of Kimchi → 김치의 나라 | 다큐멘터리
  A Star Shoots Across the Sky → Natchathiram Nagargirathu | 코미디|드라마|로맨스

유사도 낮음 중 연도 검증 가능 5건 중 일치 100%

최종 채택: 814편 (오매칭 의심 0편 제외)
CSV 갱신 완료 (오매칭 의심 제거)


**원본 코드** — `scripts/03_justwatch_dedupe.py`

In [4]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 03. JustWatch 수집 CSV 중복 정리 (중간 저장 로직의 중복 제거, 마지막 상태 유지)
#        + 장르 분포 요약
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1 — requirements.txt 참고
#        실행: python scripts/03_justwatch_dedupe.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
OUT = NBOUT / "data/processed/genre_missing_justwatch.csv"

df = pd.read_csv(OUT)
print("정리 전:", len(df), "행")
df = df.drop_duplicates(subset="show_title", keep="last")
df = df.sort_values("show_title").reset_index(drop=True)
df.to_csv(OUT, index=False, encoding="utf-8-sig")
print("정리 후:", len(df), "행")
print("매칭 성공:", int(df.has_result.sum()), f"({df.has_result.mean()*100:.0f}%)")

print("\n== 장르 분포 (매칭된 것들, shortName 기준) ===")
g = df[df.has_result].genres_short.str.split("|").explode()
print(g.value_counts().head(15).to_string())

print("\n== 매칭 실패 샘플 ==")
fail = df[~df.has_result].show_title.tolist()
print(fail[:15])

print("\n== 매칭 성공 샘플 ==")
s = df[df.has_result].head(8)
for _, r in s.iterrows():
    print(f"  {r.show_title} → {r.matched_title} ({r.matched_year}) | {r.genres_ko}")


정리 전: 814 행
정리 후: 814 행
매칭 성공: 814 (100%)

== 장르 분포 (매칭된 것들, shortName 기준) ===
genres_short
drm    384
cmy    318
act    223
trl    198
crm    149
rma    114
fml    101
ani     97
fnt     86
doc     72
rly     64
scf     63
hrr     55
hst     31
eur     30

== 매칭 실패 샘플 ==
[]

== 매칭 성공 샘플 ==
  #AtFirstSight → 사랑이 머무는 풍경 (1999.0) | 드라마|로맨스
  #Mom, I’m coming home → 커밍 홈 인 더 다크 (2021.0) | 범죄|공포|스릴러
  24 Hours in Aldi → 24 Hours in Aldi (2024.0) | 다큐멘터리
  30 Days Left → 미션 파서블 인 30 데이즈 (2020.0) | 액션|코미디|범죄|유럽에서 제작
  30 for 30: O.J.: Made in America → O.J : 메이드 인 아메리카 (2016.0) | 다큐멘터리
  5 Ways To Socially Exclude My Husband → 남편을 사회적으로 말살하는 다섯 가지 방법 (2023.0) | 드라마|스릴러
  8 Vasantalu → 8 Vasantalu (2025.0) | 드라마|로맨스
  A Different World (1987) → A Different World (1987.0) | 코미디|가족


**원본 코드** — `scripts/04_catalog_genre_update.py`

In [5]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 04. JustWatch 장르를 카탈로그에 반영 (IMDb 장르 코드 매핑)
#        + 전체 장르 커버리지 재계산 + 언어장벽 장르별 통과율 재계산
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/04_catalog_genre_update.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

# JustWatch shortName → IMDb 장르 태그 매핑
JW2IMDB = {
    "drm": "Drama", "cmy": "Comedy", "act": "Action", "trl": "Thriller",
    "crm": "Crime", "rma": "Romance", "fml": "Family", "ani": "Animation",
    "fnt": "Fantasy", "doc": "Documentary", "rly": "Reality-TV", "scf": "Sci-Fi",
    "hrr": "Horror", "hst": "History", "wes": "Western", "war": "War",
    "spo": "Sport", "msc": "Music", "mys": "Mystery", "bio": "Biography",
    "adn": "Adventure", "kid": "Kids",
}
NOT_GENRE = {"eur"}  # 유럽에서 제작 — 장르 아님

jw = pd.read_csv(ROOT / "data/processed/genre_missing_justwatch.csv")
jw = jw[jw.has_result]

def jw_to_imdb(s):
    out = []
    for c in str(s).split("|"):
        c = c.strip()
        if c in NOT_GENRE:
            continue
        if c in JW2IMDB:
            out.append(JW2IMDB[c])
    return ",".join(sorted(set(out)))

jw["genres_filled"] = jw.genres_short.map(jw_to_imdb)
n_unmapped = (jw.genres_filled == "").sum()
print(f"매핑 성공: {(jw.genres_filled!='').sum()}편 | 매핑 실패(미등록 코드만): {n_unmapped}편")

# ---- catalog 갱신 ----
cat = pd.read_csv(ROOT / "data/interim/netflix_titles_catalog.csv")
cat["genre_source"] = cat.imdb_genres.fillna("").astype(str).str.strip().ne("").map({True: "imdb", False: ""})
fill_map = dict(zip(jw.show_title, jw.genres_filled))
mask = cat.imdb_genres.fillna("").astype(str).str.strip().eq("") & cat.show_title.isin(fill_map)
cat.loc[mask, "imdb_genres"] = cat.loc[mask, "show_title"].map(fill_map)
cat.loc[mask & cat.imdb_genres.ne(""), "genre_source"] = "justwatch"
cat.to_csv(ROOT / "data/interim/netflix_titles_catalog.csv", index=False, encoding="utf-8-sig")
print(f"catalog 갱신: {mask.sum()}편 채움 | genre_source 열 추가")

# ---- 커버리지 재계산 ----
c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
c_all = pd.concat([c_old, c_new]).drop_duplicates()
titles = sorted(c_all.show_title.unique())
gen_all = cat.set_index("show_title").imdb_genres.fillna("").astype(str).str.strip()
cov = sum(1 for t in titles if gen_all.get(t, "")) / len(titles) * 100
print(f"\n장르 커버리지: {cov:.1f}% (기존 93.2%) — 미확보 {sum(1 for t in titles if not gen_all.get(t,''))}편")

# ---- 언어장벽 장르표 재계산 (결측분 반영) ----
g_old = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=",")
g_new = pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")
g_all = pd.concat([g_old, g_new]).drop_duplicates()
EN_MARKETS = {"United States", "United Kingdom", "Canada", "Australia", "Ireland", "New Zealand"}
hits = set(g_all.show_title.unique())
g_all["lang_cat"] = g_all.category.str.contains("Non-English").map({True: "비영어", False: "영어"})
lang_sets = g_all.groupby("show_title").lang_cat.apply(set)
only_ne = {t for t, s in lang_sets.items() if s == {"비영어"}}
no_country = hits - set(c_all.show_title.unique())
pop_ne = only_ne - no_country
ever = c_all.groupby("show_title").country_name.apply(set)
entered_ne = ever.loc[ever.index.isin(pop_ne)].apply(lambda cs: bool(cs & EN_MARKETS))

ne_titles = pd.Series(sorted(pop_ne), name="show_title")
df_ne = pd.DataFrame({"show_title": ne_titles.values})
df_ne["genres"] = df_ne.show_title.map(gen_all)
df_ne["entered_en"] = df_ne.show_title.map(entered_ne)
df_ne_cov = df_ne.genres.str.strip().ne("").mean() * 100
print(f"비영어 표본 {len(pop_ne)}편의 장르 커버: {df_ne_cov:.1f}%")
df_ne = df_ne[df_ne.genres.str.strip() != ""]
df_ne["장르"] = df_ne.genres.apply(lambda s: [t.strip() for t in s.split(",") if t.strip()])
ex = df_ne.explode("장르")
gstat = ex.groupby("장르").agg(표본=("entered_en", "size"), 영어권진입률=("entered_en", "mean"))
gstat["영어권진입률"] = (gstat.영어권진입률 * 100).round(1)
gstat = gstat[gstat.표본 >= 50].sort_values("영어권진입률", ascending=False).reset_index()
gstat[["장르", "표본", "영어권진입률"]].to_csv(NBOUT / "data/processed/language_barrier_by_genre.csv", index=False, encoding="utf-8-sig")
print("\n== 재계산된 장르별 영어권 통과율 ==")
print(gstat.to_string(index=False))


매핑 성공: 814편 | 매핑 실패(미등록 코드만): 0편
catalog 갱신: 0편 채움 | genre_source 열 추가



장르 커버리지: 99.7% (기존 93.2%) — 미확보 43편


비영어 표본 1824편의 장르 커버: 100.0%

== 재계산된 장르별 영어권 통과율 ==
         장르   표본  영어권진입률
    Mystery  184    33.7
   Thriller  292    31.8
      Crime  472    31.6
  Adventure  153    30.1
     Action  429    28.9
Documentary   90    28.9
    History   82    26.8
      Drama 1153    25.6
  Biography   74    23.0
     Horror  123    22.0
     Family   63    17.5
  Animation   93    17.2
    Fantasy  112    17.0
    Romance  315    15.9
     Comedy  577    14.7


**원본 코드** — `scripts/06_genre_missing_signal_check.py`

In [6]:
# -*- coding: utf-8 -*-
# ─────────────────────────────────────────────────────────────────────────────
# [용도] 06. 장르 결측 그룹 vs 장르 확보 그룹 실측 비교
#        (글로벌 히트율 10.9% vs 29.9%, 평균 데뷔 규모 3.8 vs 15.1개국)
#        — 장르 "결측"이 무작위가 아니라 저인지도 작품을 대변하는 신호임을 검증
# [환경] Python 3.15.2 (Windows) / pandas 3.1.1, numpy 2.5.4 — requirements.txt 참고
#        실행: python scripts/06_genre_missing_signal_check.py
# [경로] ROOT = 스크립트 위치 기준 프로젝트 루트 (scripts/ 상위)
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")

c_old = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", sep=",", encoding="cp1252")
c_new = pd.read_csv(ROOT / "data/raw/all-weeks-countries_2026-11-11.tsv", sep="\t")
c_all = pd.concat([c_old, c_new]).drop_duplicates()
g_all = pd.concat([
    pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", sep=","),
    pd.read_csv(ROOT / "data/raw/all-weeks-global_2026-11-11.tsv", sep="\t")]).drop_duplicates()

hits = set(g_all.show_title.unique())
cat = pd.read_csv(ROOT / "data/interim/netflix_titles_catalog.csv")
gen = cat.set_index("show_title").imdb_genres.fillna("").astype(str).str.strip()

titles = pd.DataFrame({"show_title": sorted(c_all.show_title.unique())})
titles["has_genre"] = titles.show_title.map(gen).fillna("").str.strip().ne("")
titles["is_hit"] = titles.show_title.isin(hits).astype(int)
cnt = c_all.groupby("show_title").country_name.nunique()
titles["국가수"] = titles.show_title.map(cnt)

g = titles.groupby("has_genre").agg(
    작품수=("show_title", "size"),
    글로벌히트율=("is_hit", "mean"),
    평균진입국가수=("국가수", "mean"))
g.index = ["결측(장르 미확보)", "장르 확보"]
g["글로벌히트율"] = (g["글로벌히트율"] * 100).round(1)
g["평균진입국가수"] = g["평균진입국가수"].round(1)
print(g.to_string())


              작품수  글로벌히트율  평균진입국가수
결측(장르 미확보)     43     2.3      2.4
장르 확보       12443    28.7     14.4


### 실행 결과 해설 — 보고서 수치와의 관계

- **커버리지**: 카탈로그 갱신 결과 **99.7%(기존 93.2%)·미확보 43편** — 보고서 01장 수치와 일치한다.
- **결측 신호 검증 수치가 다른 이유**: 보고서의 "결측 작품 히트율 10.9% vs 확보 29.9%"는 **JustWatch 보완 전(결측 853편 시점)** 비교다.
  이 노트북은 보완이 이미 끝난 현재 데이터로 다시 실행했기 때문에 결측 43편 vs 확보 12,443편 기준(2.3% vs 28.7%)으로 나온다.
  즉 코드는 동일하고, **재실행 시점의 데이터 상태가 다른 것**이다. 853편 시점의 원래 비교는 제출 당시 실행 기록에 있다.